# Prioritized Experience Replay — Schaul et al. (2015)

_Papers · Reinforcement Learning_

**Replay the transitions you learn most from, not just the ones you happened to see.**

---

This notebook is a practice scaffold for the **Prioritized Experience Replay — Schaul et al. (2015)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, numpy as np
import gym   # CartPole; preinstalled in Colab. (If missing, see the CODEVIZ cell -- no install needed.)

torch.manual_seed(0); np.random.seed(0)

# --- 0. Recompute the lesson's worked example: priorities -> probs -> IS-weights. ---
eps, alpha, beta, N = 0.01, 0.6, 0.4, 3
d   = np.array([2.0, 1.0, 0.5])              # |TD-errors|
p   = np.abs(d) + eps                        # p_i = |delta_i| + eps
P   = p**alpha / (p**alpha).sum()            # Eqn. 1:  P(i) = p_i^a / sum_k p_k^a
w   = (1.0 / (N * P))**beta                  # raw IS-weight  w_i = (1/(N P(i)))^beta
wn  = w / w.max()                            # normalize by max -> only scales down
print("P  =", np.round(P, 4).tolist())       # [0.476, 0.315, 0.209]
print("w/max =", np.round(wn, 4).tolist())   # [0.7195, 0.8487, 1.0]
# sanity: uniform (alpha=0) gives P=1/3 and all weights exactly 1
print("uniform w =", float(((1/(N*(1/N)))**beta)))   # 1.0


# --- 1. The prioritized replay buffer (built by hand; alpha=0 => uniform ablation). ---
class PrioritizedReplay:
    def __init__(self, cap, alpha=0.6, eps=1e-2):
        self.cap, self.alpha, self.eps = cap, alpha, eps
        self.data = []; self.prio = np.zeros(cap, dtype=np.float32); self.pos = 0
    def add(self, *transition):
        maxp = self.prio.max() if self.data else 1.0          # new -> MAX priority
        if len(self.data) < self.cap: self.data.append(transition)
        else: self.data[self.pos] = transition
        self.prio[self.pos] = maxp
        self.pos = (self.pos + 1) % self.cap
    def sample(self, batch, beta):
        n = len(self.data)
        pa = self.prio[:n] ** self.alpha                      # p_i^alpha   (alpha=0 -> all ones -> uniform)
        Pi = pa / pa.sum()                                    # Eqn. 1
        idx = np.random.choice(n, batch, p=Pi)
        w = (n * Pi[idx]) ** (-beta)                          # w_i = (N P(i))^(-beta)
        w = w / w.max()                                       # normalize -> downscale only
        b = [self.data[i] for i in idx]
        return idx, b, torch.tensor(w, dtype=torch.float32)
    def update(self, idx, td):                                # write |delta|+eps back
        self.prio[idx] = np.abs(td) + self.eps


def qnet(obs, act):
    return nn.Sequential(nn.Linear(obs, 128), nn.ReLU(), nn.Linear(128, act))


def train(prioritized, episodes=200, gamma=0.99):
    torch.manual_seed(0); np.random.seed(0)
    env = gym.make("CartPole-v1")
    o_dim = env.observation_space.shape[0]; a_dim = env.action_space.n
    q, qt = qnet(o_dim, a_dim), qnet(o_dim, a_dim); qt.load_state_dict(q.state_dict())
    opt = torch.optim.Adam(q.parameters(), lr=1e-3)
    buf = PrioritizedReplay(10000, alpha=(0.6 if prioritized else 0.0))
    eps_greedy, returns = 1.0, []
    for ep in range(episodes):
        s, _ = env.reset(seed=ep); done = False; R = 0.0
        beta = 0.4 + 0.6 * ep / episodes                      # anneal beta 0.4 -> 1.0
        while not done:
            a = env.action_space.sample() if np.random.rand() < eps_greedy \
                else int(q(torch.tensor(s, dtype=torch.float32)).argmax())
            s2, r, term, trunc, _ = env.step(a); done = term or trunc
            buf.add(s, a, r, s2, done); s = s2; R += r
            if len(buf.data) >= 64:
                idx, batch, w = buf.sample(64, beta)
                S  = torch.tensor(np.array([b[0] for b in batch]), dtype=torch.float32)
                A  = torch.tensor([b[1] for b in batch])
                Rw = torch.tensor([b[2] for b in batch], dtype=torch.float32)
                S2 = torch.tensor(np.array([b[3] for b in batch]), dtype=torch.float32)
                D  = torch.tensor([b[4] for b in batch], dtype=torch.float32)
                qsa = q(S).gather(1, A[:, None]).squeeze(1)
                with torch.no_grad():
                    tgt = Rw + gamma * qt(S2).max(1).values * (1 - D)
                td = tgt - qsa
                loss = (w * td.pow(2)).mean()                 # IS-weighted loss
                opt.zero_grad(); loss.backward(); opt.step()
                buf.update(idx, td.detach().numpy())          # priorities <- |delta|+eps
        eps_greedy = max(0.05, eps_greedy * 0.97)
        if ep % 5 == 0: qt.load_state_dict(q.state_dict())
        returns.append(R)
    env.close(); return returns

print("\nPRIORITIZED replay:")
pr = train(prioritized=True)
print("UNIFORM replay (ablation, alpha=0):")
un = train(prioritized=False)
print("\nprioritized mean return (last 50 eps):", round(np.mean(pr[-50:]), 1))
print("uniform     mean return (last 50 eps):", round(np.mean(un[-50:]), 1))
# Prioritized typically reaches high return in fewer episodes than uniform.
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_Does prioritizing high-TD-error transitions learn with fewer updates than uniform replay -- and does the gap grow with task length?_

In [ ]:
import numpy as np

# Blind Cliffwalk (paper Sec 2 / Fig 1): chain of n states. Each state has two actions:
#   action 'right' (a=1) advances to the next state; action 'wrong' (a=0) terminates with 0.
#   Reaching the end gives reward 1 (terminal). gamma=1; only the last transition is rewarding.
# Collect every transition once into a fixed buffer, then run Q-learning updates, sampling either
# UNIFORMLY or by PRIORITY (oracle = current max |TD-error|). Count updates to reach Q*.
def build(n):
    t = []
    for s in range(n):
        t.append((s, 0, 0.0, None, True))                       # wrong -> terminate, r=0
        t.append((s, 1, 1.0, None, True) if s == n - 1
                 else (s, 1, 0.0, s + 1, False))                # right -> advance (last -> r=1)
    return t

def run(n, mode, seed=0, gamma=1.0, max_updates=10_000_000):
    rng = np.random.default_rng(seed); trans = build(n)
    Q = np.zeros((n, 2))
    def td(t):
        s, a, r, sn, done = t
        return (r if done else r + gamma * Q[sn].max()) - Q[s, a]
    Qstar = np.zeros((n, 2)); Qstar[:, 1] = 1.0                 # optimal: Q*(s,right)=1, Q*(s,wrong)=0
    for u in range(1, max_updates + 1):
        if mode == "uniform":
            i = rng.integers(len(trans))                       # sample blindly
        else:
            e = np.abs([td(t) for t in trans])                 # priority oracle: max |TD-error|
            i = int(rng.choice(np.flatnonzero(e >= e.max() - 1e-12)))
        t = trans[i]; Q[t[0], t[1]] += td(t)                   # exact tabular update (lr=1)
        if np.max(np.abs(Q - Qstar)) < 1e-9:
            return u

ns = [4, 6, 8, 10, 12, 14, 16]
unif = [int(np.median([run(n, "uniform",  s) for s in range(15)])) for n in ns]
prio = [int(np.median([run(n, "priority", s) for s in range(15)])) for n in ns]
print("uniform     :", [[n, u] for n, u in zip(ns, unif)])
print("prioritized :", [[n, p] for n, p in zip(ns, prio)])
print("speedup     :", [round(u / p, 1) for u, p in zip(unif, prio)])
# uniform     -> 31, 64, 105, 156, 274, 384, 471   (grows fast)
# prioritized -> n exactly:  4, 6, 8, 10, 12, 14, 16
# speedup grows with n (~7x -> ~28x): the paper's exponential Blind-Cliffwalk effect.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a working prioritized DQN on a sparse-reward task (only the final
            transition gives reward). You flip alpha from $0.6$ to $0$ and retrain with everything
            else identical. What happens to learning speed, and what does that isolate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set alpha = 0. Now $p_i^0 = 1$ for every transition, so $P(i) = 1/N$ &mdash; the buffer samples uniformly. — _$\alpha$ is the single knob between prioritized and uniform; turning it to $0$ recovers the original DQN replay exactly._
- Retrain and watch updates-to-converge: uniform needs many more, because the one informative (reward-reaching) transition is now sampled as rarely as every boring one. — _Uniform spends most updates on already-learned transitions; the rare high-TD-error transition that carries the signal is diluted._
- Note the gap grows with task length (more states to chain the reward back through). — _Each extra state multiplies the number of uniform draws needed to hit the right transition in the right order &mdash; the paper's exponential Blind-Cliffwalk effect._

**Answer:** With $\alpha = 0$ the buffer is uniform and learning is much slower &mdash; and the slowdown
                 worsens as the task lengthens. Since only $\alpha$ changed, this isolates prioritized
                 sampling (not network size, optimizer, or data) as the cause of the speed-up. The CODEVIZ
                 panel reproduces exactly this contrast on the Blind Cliffwalk.

</details>

**Problem 2.** Recompute the pipeline for a fresh batch. Buffer of $N = 2$ transitions with $|\delta| = [3.0,\;
            1.0]$, $\epsilon = 0$, $\alpha = 0.5$, $\beta = 1$. Find $P(i)$ and the normalized IS-weights.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Priorities (no floor): $p = [3.0,\; 1.0]$. Raise to $\alpha = 0.5$ (square root): $p^{0.5} = [\sqrt{3},\; \sqrt{1}] = [1.732,\; 1.0]$, sum $= 2.732$. — _$P(i) \propto p_i^{\alpha}$ needs the powered priorities and their sum (Eqn. 1)._
- Probabilities: $P = [1.732/2.732,\; 1.0/2.732] = [0.634,\; 0.366]$. — _Normalize so they sum to $1$._
- Raw weights $w_i = (1/(N\,P(i)))^{\beta} = (1/(2\,P(i)))^{1}$: $w = [1/(2\cdot0.634),\; 1/(2\cdot0.366)] = [0.789,\; 1.366]$. — _$\beta = 1$ is full correction, so $w_i = 1/(N P(i))$ directly._
- Normalize by $\max w = 1.366$: $w/\max = [0.577,\; 1.0]$. — _Largest weight becomes $1$; updates only scale down._

**Answer:** $P = [0.634,\; 0.366]$; normalized IS-weights $= [0.577,\; 1.0]$. The high-error transition
                 is sampled more (P = 0.634) but its update is scaled down most (weight 0.577) &mdash;
                 prioritize the sampling, then undo the resulting over-count in the gradient.

</details>

**Problem 3.** A teammate removes the $\epsilon$ floor "to simplify," using $p_i = |\delta_i|$. After a while
            some transitions stop being replayed entirely, even ones that later turn out to be mispredicted.
            Why, and what is the fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Find a transition the network currently predicts perfectly: $\delta = 0 \Rightarrow p_i = |0| = 0$. — _Priority is the absolute TD-error, which can hit exactly $0$._
- Its sampling probability $P(i) = p_i^{\alpha} / \sum_k p_k^{\alpha} = 0$, so it is never drawn again. — _A zero priority gives zero probability under Eqn. 1 &mdash; permanent exclusion._
- If the network later drifts and that transition becomes wrong, its priority is still recorded as $0$, so it stays frozen out and the error is never corrected. — _Priorities only update when a transition is sampled; a never-sampled transition can never refresh its stale $0$._
- Fix: restore the floor, $p_i = |\delta_i| + \epsilon$ with small $\epsilon$. — _A nonzero floor guarantees every transition keeps a small but positive chance of being revisited._

**Answer:** Without $\epsilon$, any transition reaching $\delta = 0$ gets priority $0$ and probability
                 $0$, so it is frozen out permanently &mdash; and can never refresh its priority even if it
                 later becomes mispredicted. The $\epsilon$ floor ($p_i = |\delta_i| + \epsilon$) keeps every
                 transition reachable.

</details>